# Loss Prediction Model

This notebook builds a reviewer-friendly baseline for predicting whether a record is likely to be **loss-making**. The emphasis is on concise exploration, leakage-aware features, and a small model comparison set that is still easy to explain.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "README.md").exists() and (PROJECT_ROOT.parent / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from pycaret.classification import (
    compare_models,
    finalize_model,
    get_config,
    predict_model,
    pull,
    setup,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

from utils.config.config import Config
from utils.ml.pickle_handler import write_to_pickle

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

classification_cfg = Config(single_config_path=str(PROJECT_ROOT / "conf" / "classification.toml")).get("modeling")
required_cfg_keys = {
    "target_column",
    "feature_columns",
    "categorical_features",
    "numeric_features",
    "eda_numeric_features",
    "candidate_models",
    "session_id",
    "train_size",
    "fold",
    "sort_metric",
}
missing_cfg_keys = sorted(required_cfg_keys - set(classification_cfg or {}))
if missing_cfg_keys:
    raise KeyError(f"Missing classification config keys: {missing_cfg_keys}")

TARGET = classification_cfg["target_column"]
BASE_FEATURES = classification_cfg["feature_columns"]
CATEGORICAL_FEATURES = classification_cfg["categorical_features"]
NUMERIC_FEATURES = classification_cfg["numeric_features"]
EDA_NUMERIC_FEATURES = classification_cfg["eda_numeric_features"]
ARTIFACT_PATH = PROJECT_ROOT / "app" / "artifacts" / "model.pkl"
DATA_PATH = PROJECT_ROOT / "data" / "records.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
raw_labels = df[TARGET].astype("string").str.strip().str.lower()
df[TARGET] = (
    raw_labels
    .replace({"": pd.NA, "nan": pd.NA, "<na>": pd.NA})
    .map({"true": True, "false": False})
)
invalid_labels = raw_labels[df[TARGET].isna()].dropna().unique().tolist()
invalid_label_count = len(invalid_labels)
if invalid_label_count:
    raise ValueError(f"Unexpected labels in {TARGET}: {invalid_labels}")
if df[TARGET].isna().any():
    raise ValueError(f"Missing labels found in {TARGET}: {int(df[TARGET].isna().sum())}")
df[TARGET] = (
    df[TARGET]
    .astype(bool)
)

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from {DATA_PATH.relative_to(PROJECT_ROOT)}")
df.head()

## 1. Data Exploration

The goal here is to surface the specific modelling risks in the raw file: class imbalance, noisy categoricals, missingness, and leakage from `loss_ratio`.

In [ ]:
dtype_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=False),
    }
).sort_values(["missing_count", "n_unique"], ascending=[False, False])

dtype_summary

In [ ]:
class_balance = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="count")
)
class_balance["pct"] = (class_balance["count"] / len(df) * 100).round(2)

class_balance

In [ ]:
display(df[EDA_NUMERIC_FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.ravel()
for index, column in enumerate(EDA_NUMERIC_FEATURES):
    sns.histplot(df[column], kde=True, ax=axes[index])
    axes[index].set_title(column)
axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for feature in CATEGORICAL_FEATURES:
    cleaned_feature = (
        df[feature]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({"": pd.NA, "nan": pd.NA, "<na>": pd.NA})
        .fillna("unknown")
    )
    summary = (
        df.assign(**{feature: cleaned_feature})
        .groupby(feature, dropna=False)[TARGET]
        .agg(["count", "mean"])
        .rename(columns={"mean": "loss_rate"})
        .sort_values(["loss_rate", "count"], ascending=[False, False])
    )
    summary["loss_rate"] = summary["loss_rate"].round(3)
    print(f"\n{feature} loss-rate summary")
    display(summary.head(15))

In [ ]:
correlation_df = df.copy()
correlation_df[TARGET] = correlation_df[TARGET].astype(int)
numeric_corr = correlation_df[EDA_NUMERIC_FEATURES + [TARGET]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_corr.round(2), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Numeric feature correlations")
plt.show()

numeric_corr[[TARGET]].sort_values(by=TARGET, ascending=False)

Key findings:
- `loss_ratio` is tightly coupled to the label and must be excluded from model features to avoid leakage.
- The dataset is imbalanced, so raw accuracy alone would overstate model quality.
- Categorical quality issues are real but lightweight: casing, whitespace, and blank values are the main cleanup needs.
- Exposure and claims history fields look directionally useful enough for a reviewer-support baseline.

## 2. Feature Engineering

The feature set stays compact and explainable. Cleaning is limited to obvious data-quality issues, and the only derived features are simple rate-style terms a reviewer can reason about.

In [ ]:
clean_df = df.copy()

for column in NUMERIC_FEATURES + ["loss_ratio"]:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

for column in CATEGORICAL_FEATURES:
    clean_df[column] = (
        clean_df[column]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({"": pd.NA, "nan": pd.NA, "<na>": pd.NA})
        .fillna("unknown")
    )

clean_df["prior_claims"] = clean_df["prior_claims"].fillna(0).clip(lower=0)
clean_df["years_trading"] = clean_df["years_trading"].clip(lower=0)
clean_df["years_trading"] = clean_df["years_trading"].fillna(clean_df["years_trading"].median())
clean_df["premium"] = clean_df["premium"].fillna(clean_df["premium"].median()).clip(lower=0)
clean_df["limit"] = clean_df["limit"].fillna(clean_df["limit"].median()).clip(lower=1)
clean_df[TARGET] = clean_df[TARGET].astype(bool)

clean_df[BASE_FEATURES + [TARGET]].head()

In [ ]:
print(
    "Categorical encoding will be handled inside PyCaret setup so the saved artifact keeps "
    "preprocessing and prediction together."
)

In [ ]:
clean_df["premium_rate"] = clean_df["premium"] / clean_df["limit"]
clean_df["claims_per_year"] = clean_df["prior_claims"] / clean_df["years_trading"].clip(lower=1)

leakage_guard_df = clean_df.drop(columns=["record_id", "loss_ratio", "is_loss_making"])

model_features = BASE_FEATURES + ["premium_rate", "claims_per_year"]
model_df = leakage_guard_df[model_features].copy()
model_df[TARGET] = clean_df[TARGET].astype(bool)

X = model_df[model_features]
y = model_df[TARGET].astype(bool)

display(model_df.head())
print("Model features:", model_features)
print("Target positive rate:", round(y.mean(), 3))

`loss_ratio` is excluded from `X` because it directly determines `is_loss_making` and would leak the answer into training.

## 3. Model Training

PyCaret is used here as a structured comparison layer, but the candidate set is deliberately small so the result remains reviewer-friendly and notebook runtime stays manageable.

In [ ]:
setup_kwargs = {
    "data": model_df,
    "target": TARGET,
    "session_id": classification_cfg["session_id"],
    "train_size": classification_cfg["train_size"],
    "fold": classification_cfg["fold"],
    "n_jobs": 1,
    "log_experiment": False,
    "html": False,
}

try:
    experiment = setup(silent=True, **setup_kwargs)
except TypeError:
    experiment = setup(verbose=False, **setup_kwargs)

experiment

In [ ]:
expected_candidate_models = ["lr", "dt", "rf", "et", "lightgbm"]
if classification_cfg["candidate_models"] != expected_candidate_models:
    raise ValueError(
        f"Unexpected candidate_models config: {classification_cfg['candidate_models']}"
    )

best_model = compare_models(
    include=["lr", "dt", "rf", "et", "lightgbm"],
    sort=classification_cfg["sort_metric"],
    turbo=True,
)
comparison_results = pull()

display(comparison_results)
print("Selected baseline:", type(best_model).__name__)
best_model

The comparison set is intentionally narrow: logistic regression and a single decision tree anchor interpretability, while random forest, extra trees, and LightGBM test whether modestly more complex tree ensembles earn their keep.

In [ ]:
final_model = finalize_model(best_model)
final_model

In [ ]:
holdout_predictions = predict_model(best_model)
display(holdout_predictions.head())

score_column = next(
    (column for column in ["Score", "prediction_score", "prediction_score_1"] if column in holdout_predictions.columns),
    None,
)
label_column = next(
    (column for column in ["prediction_label", "Label"] if column in holdout_predictions.columns),
    None,
)
if score_column is None or label_column is None:
    raise KeyError(f"Unexpected predict_model columns: {holdout_predictions.columns.tolist()}")

def to_binary(series: pd.Series) -> pd.Series:
    if str(series.dtype) == "bool":
        return series.astype(int)
    mapped = series.astype(str).str.lower().map({"true": 1, "false": 0, "1": 1, "0": 0})
    if mapped.notna().all():
        return mapped.astype(int)
    return series.astype(int)

y_true = to_binary(holdout_predictions[TARGET])
y_pred = to_binary(holdout_predictions[label_column])
y_score = holdout_predictions[score_column].astype(float)

print(classification_report(y_true, y_pred, digits=3))
print("Holdout ROC-AUC:", round(roc_auc_score(y_true, y_score), 4))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap="Blues")
plt.title("Holdout confusion matrix")
plt.show()

In [ ]:
model_step = final_model
if hasattr(final_model, "named_steps"):
    for step_name in ["trained_model", "actual_estimator", "classifier"]:
        if step_name in final_model.named_steps:
            model_step = final_model.named_steps[step_name]
            break

feature_names = list(model_features)
for config_name in ["X_train_transformed", "X_transformed", "X_train"]:
    try:
        feature_frame = get_config(config_name)
        if hasattr(feature_frame, "columns"):
            feature_names = list(feature_frame.columns)
            break
    except Exception:
        continue

if hasattr(model_step, "feature_importances_"):
    importance_values = model_step.feature_importances_
    if len(feature_names) != len(importance_values):
        feature_names = [f"feature_{index}" for index in range(len(importance_values))]
    importance_df = pd.DataFrame(
        {"feature": feature_names, "importance": importance_values}
    ).sort_values("importance", ascending=False)
    display(importance_df.head(10))
    plt.figure(figsize=(8, 4))
    sns.barplot(data=importance_df.head(10), x="importance", y="feature", orient="h")
    plt.title("Top feature importances")
    plt.show()
elif hasattr(model_step, "coef_"):
    coefficient_values = model_step.coef_[0]
    if len(feature_names) != len(coefficient_values):
        feature_names = [f"feature_{index}" for index in range(len(coefficient_values))]
    coefficient_df = pd.DataFrame(
        {"feature": feature_names, "coefficient": coefficient_values}
    ).sort_values("coefficient", key=lambda series: series.abs(), ascending=False)
    display(coefficient_df.head(10))
else:
    print("Model-specific importances are not directly available for this estimator.")

This is a reviewer-support model, not an autonomous decision system. The training data is small and intentionally noisy, so the score should be treated as one input into triage rather than a final answer.

## 4. Save Model Artifact

The saved artifact includes the finalized model plus the feature metadata and config needed to reproduce the modelling context downstream.

In [ ]:
artifact = {
    "model": final_model,
    "feature_columns": model_features,
    "config": classification_cfg,
    "target_column": TARGET,
}

write_to_pickle(artifact, str(ARTIFACT_PATH))
print(f"Model saved to {ARTIFACT_PATH}")

In [ ]:
print("Artifact exists:", ARTIFACT_PATH.exists())
if ARTIFACT_PATH.exists():
    print("Artifact size (bytes):", ARTIFACT_PATH.stat().st_size)
ARTIFACT_PATH